# **Feature Engineering**

This notebook begins the feature engineering phase of the HDFS_v1 anomaly detection study.

The exploratory analysis established an understanding of the dataset structure, validated relationships between the primary data sources, and identified the information available for anomaly detection. Building upon those findings, this notebook transforms the raw HDFS data into structured feature sets that will be used throughout the experimental phase of the study.

Feature engineering is the process of converting raw data into numerical features that machine learning algorithms and statistical methods can use for learning and prediction. Rather than training directly on the original datasets, this study constructs multiple feature representations to investigate which characteristics of HDFS block processing provide the greatest predictive value for anomaly detection.

Four feature sets will be developed:

1. **Feature Set 1 – Event Occurrence Features**
   - Event occurrence counts (E1 through E29).

2. **Feature Set 2 – Event Occurrence + Basic Trace Features**
   - Event occurrence counts.
   - Trace length.
   - Number of unique events within each trace.

3. **Feature Set 3 – Event Occurrence + Basic Trace Features + Two-Event Sequence Features**
   - Event occurrence counts.
   - Basic trace features.
   - Ordered two-event sequence counts.

4. **Feature Set 4 – Event Occurrence + Basic Trace Features + Three-Event Sequence Features**
   - Event occurrence counts.
   - Basic trace features.
   - Ordered three-event sequence counts.

Each feature set will be exported as an independent processed dataset and evaluated using the same experimental framework. This approach allows the study to determine whether progressively richer feature representations improve anomaly detection performance.

The engineered feature sets will later be evaluated using:

- A statistical rule-based baseline.
- Logistic Regression.
- Random Forest.
- Isolation Forest.

Model performance will be compared using Accuracy, Precision, Recall, and F1 Score.

The objective of this notebook is not to train models, but rather to construct consistent, reproducible feature sets that serve as the foundation for every experiment performed throughout this research. 

### *Feature Set 1: Event Occurrence Features*

Feature Set 1 uses the event occurrence matrix as the base feature representation. This feature set includes the E1 through E29 columns, where each column represents the number of times a specific HDFS event occurred during processing of a block.

The `BlockId` column is retained in the exported dataset for tracking and reference purposes, but it will not be used as a model input. This identifier is useful for tracing predictions back to the original HDFS block if additional investigation is needed.

The `Label` column is retained as the ground-truth target variable for training and evaluation.

The `Type` column is excluded because it contains anomaly-specific information and is populated only for failed blocks. Including this column as a model feature would introduce target leakage and create an unfair evaluation.

The resulting dataset will be exported to the `data/processed/` folder as `feature_set_1.csv`.

In [2]:
# Import required libraries

from pathlib import Path

import pandas as pd
import kagglehub

In [3]:
# Set project paths

current_path = Path.cwd()

if current_path.name == "notebooks":
    project_root = current_path.parent
else:
    project_root = current_path

data_dir = project_root / "data"
processed_dir = data_dir / "processed"

processed_dir.mkdir(parents=True, exist_ok=True)

print("Project root:", project_root)
print("Processed data folder:", processed_dir)

Project root: c:\Users\taman\Documents\hdfs-anomaly-detection-study
Processed data folder: c:\Users\taman\Documents\hdfs-anomaly-detection-study\data\processed


In [4]:
# Download or locate the HDFS_v1 dataset archive from Kaggle

dataset_path = Path(
    kagglehub.dataset_download(
        "tamaniwilliams/hdfs-v1-loghub-dataset-archive"
    )
)

print("Dataset location:")
print(dataset_path)

Dataset location:
C:\Users\taman\.cache\kagglehub\datasets\tamaniwilliams\hdfs-v1-loghub-dataset-archive\versions\1


In [5]:
# Load the source datasets needed for feature engineering

matrix_df = pd.read_csv(
    dataset_path / "Event_occurrence_matrix.csv"
)

traces_df = pd.read_csv(
    dataset_path / "Event_traces.csv"
)

print("Event occurrence matrix shape:", matrix_df.shape)
print("Event traces shape:", traces_df.shape)

Event occurrence matrix shape: (575061, 32)
Event traces shape: (575061, 6)


In [6]:
# Create Feature Set 1 using event occurrence features

feature_set_1 = matrix_df.drop(columns=["Type"])

feature_set_1.head()

,BlockId,Label,E1,E2,E3,E4,E5,E6,E7,E8,...,E20,E21,E22,E23,E24,E25,E26,E27,E28,E29
0,blk_-1608999687919862906,Success,0,0,203,0,10,7,0,0,...,0,10,1,10,0,4,10,0,0,0
1,blk_7503483334202473044,Success,0,2,1,0,3,0,0,0,...,0,3,1,3,0,0,3,0,0,0
2,blk_-3544583377289625738,Fail,0,0,203,0,3,0,0,0,...,1,3,1,3,0,0,3,0,0,0
3,blk_-9073992586687739851,Success,0,3,0,0,3,0,0,0,...,0,3,1,3,0,0,3,0,0,0
4,blk_7854771516489510256,Success,0,3,1,15,3,0,0,0,...,0,3,1,3,0,0,3,0,0,0


In [7]:
# Verify Feature Set 1 dimensions and columns

print(f"Rows: {feature_set_1.shape[0]:,}")
print(f"Columns: {feature_set_1.shape[1]}")

feature_set_1.columns.tolist()

Rows: 575,061
Columns: 31


['BlockId',
 'Label',
 'E1',
 'E2',
 'E3',
 'E4',
 'E5',
 'E6',
 'E7',
 'E8',
 'E9',
 'E10',
 'E11',
 'E12',
 'E13',
 'E14',
 'E15',
 'E16',
 'E17',
 'E18',
 'E19',
 'E20',
 'E21',
 'E22',
 'E23',
 'E24',
 'E25',
 'E26',
 'E27',
 'E28',
 'E29']

In [8]:
# Export Feature Set 1 to the processed data folder

feature_set_1_path = processed_dir / "feature_set_1.csv"

feature_set_1.to_csv(feature_set_1_path, index=False)

print("Feature Set 1 saved to:")
print(feature_set_1_path)

Feature Set 1 saved to:
c:\Users\taman\Documents\hdfs-anomaly-detection-study\data\processed\feature_set_1.csv


## Feature Set 2: Event Occurrence + Basic Trace Features

Feature Set 2 extends Feature Set 1 by adding basic characteristics from the original event traces.

In addition to the event occurrence columns E1 through E29, this feature set includes:

- **trace_length**: the total number of events in the block trace.
- **unique_event_count**: the number of different EventIds that appear in the block trace.

These features summarize the overall size and diversity of each block's event sequence without adding ordered sequence patterns yet.

The resulting dataset will be exported to the `data/processed/` folder as `feature_set_2.csv`.

In [10]:
# Create a new dataframe containing only the columns needed to calculate the additional trace features
trace_features = traces_df[["BlockId", "Features"]].copy()


# Convert the string representation of each trace into a Python list of individual EventIds
trace_features["event_list"] = trace_features["Features"].apply(
    lambda x: x.strip("[]").split(",")
)


# Count the total number of events contained within each HDFS block trace: trace length
trace_features["trace_length"] = trace_features["event_list"].apply(len)


# Count the number of unique EventIds that within each trace
trace_features["unique_event_count"] = trace_features["event_list"].apply(
    lambda x: len(set(x))
)

# Display the engineered trace features
trace_features.head()

,BlockId,Features,event_list,trace_length,unique_event_count
0,blk_-1608999687919862906,"[E5,E22,E5,E5,E11,E11,E9,E9,E11,E9,E26,E26,E26...","[E5, E22, E5, E5, E11, E11, E9, E9, E11, E9, E...",269,12
1,blk_7503483334202473044,"[E5,E5,E22,E5,E11,E9,E11,E9,E11,E9,E26,E26,E26...","[E5, E5, E22, E5, E11, E9, E11, E9, E11, E9, E...",22,9
2,blk_-3544583377289625738,"[E5,E22,E5,E5,E11,E9,E11,E9,E11,E9,E3,E26,E26,...","[E5, E22, E5, E5, E11, E9, E11, E9, E11, E9, E...",223,9
3,blk_-9073992586687739851,"[E5,E22,E5,E5,E11,E9,E11,E9,E11,E9,E26,E26,E26...","[E5, E22, E5, E5, E11, E9, E11, E9, E11, E9, E...",22,8
4,blk_7854771516489510256,"[E5,E5,E22,E5,E11,E9,E11,E9,E11,E9,E26,E26,E26...","[E5, E5, E22, E5, E11, E9, E11, E9, E11, E9, E...",38,10


In [11]:
# Merge the engineered trace features into Feature Set 1 using BlockId as the matching key

feature_set_2 = feature_set_1.merge(

    # Select only the columns needed from the engineered trace feature dataframe
    trace_features[
        [
            "BlockId",
            "trace_length",
            "unique_event_count"
        ]
    ],

    # Match rows using the BlockId column, Keep every block contained in Feature Set 1
    on="BlockId",
    how="left"
)


# Display the completed Feature Set 2
feature_set_2.head()

,BlockId,Label,E1,E2,E3,E4,E5,E6,E7,E8,...,E22,E23,E24,E25,E26,E27,E28,E29,trace_length,unique_event_count
0,blk_-1608999687919862906,Success,0,0,203,0,10,7,0,0,...,1,10,0,4,10,0,0,0,269,12
1,blk_7503483334202473044,Success,0,2,1,0,3,0,0,0,...,1,3,0,0,3,0,0,0,22,9
2,blk_-3544583377289625738,Fail,0,0,203,0,3,0,0,0,...,1,3,0,0,3,0,0,0,223,9
3,blk_-9073992586687739851,Success,0,3,0,0,3,0,0,0,...,1,3,0,0,3,0,0,0,22,8
4,blk_7854771516489510256,Success,0,3,1,15,3,0,0,0,...,1,3,0,0,3,0,0,0,38,10


In [12]:
# Verify the dimensions of Feature Set 2

print(f"Rows: {feature_set_2.shape[0]:,}")
print(f"Columns: {feature_set_2.shape[1]}")


# Display every column included in the feature set

feature_set_2.columns.tolist()

Rows: 575,061
Columns: 33


['BlockId',
 'Label',
 'E1',
 'E2',
 'E3',
 'E4',
 'E5',
 'E6',
 'E7',
 'E8',
 'E9',
 'E10',
 'E11',
 'E12',
 'E13',
 'E14',
 'E15',
 'E16',
 'E17',
 'E18',
 'E19',
 'E20',
 'E21',
 'E22',
 'E23',
 'E24',
 'E25',
 'E26',
 'E27',
 'E28',
 'E29',
 'trace_length',
 'unique_event_count']

In [13]:
# Define the output location for Feature Set 2

feature_set_2_path = (
    processed_dir / "feature_set_2.csv"
)


# Export the feature set to the processed folder

feature_set_2.to_csv(
    feature_set_2_path,
    index=False
)


# Confirm the file was successfully created

print("Feature Set 2 saved to:")
print(feature_set_2_path)

Feature Set 2 saved to:
c:\Users\taman\Documents\hdfs-anomaly-detection-study\data\processed\feature_set_2.csv


### *Feature Set 3: Event Occurrence + Basic Trace Features + Two Event Sequence Features*

Feature Set 3 extends the previous feature set by incorporating information about the ordering of events within each HDFS block trace.

While the occurrence matrix records how many times each event occurred, it does not preserve the sequence in which those events appeared. To capture some of this sequential behavior without requiring specialized sequence models, this feature set introduces **ordered two-event sequences (bigrams)**.

A two-event sequence consists of two consecutive EventIds that occur within a trace.

Examples include:

* E5 → E9
* E9 → E11
* E11 → E21

Each unique two-event sequence becomes its own feature column. The value stored within each column represents the number of times that ordered sequence appears within a given HDFS block.

Feature Set 3 therefore contains:

* Event occurrence counts (E1 through E29)
* Basic trace features
    * Trace length
    * Number of unique events
* Ordered two-event sequence counts

This feature set preserves a portion of the temporal information contained within the original event traces while remaining compatible with traditional machine learning algorithms.

In [14]:
# Import Counter for counting ordered event sequences

from collections import Counter

In [15]:
# Create ordered two-event sequences from each event trace

two_event_features = traces_df[["BlockId", "Features"]].copy()


# Convert each trace into a list of EventIds

two_event_features["event_list"] = two_event_features["Features"].apply(
    lambda x: x.strip("[]").split(",")
)


# Generate every ordered two-event sequence contained within each trace

two_event_features["bigrams"] = two_event_features["event_list"].apply(
    lambda events: [
        f"{events[i]}->{events[i+1]}"
        for i in range(len(events) - 1)
    ]
)

two_event_features.head()

,BlockId,Features,event_list,bigrams
0,blk_-1608999687919862906,"[E5,E22,E5,E5,E11,E11,E9,E9,E11,E9,E26,E26,E26...","[E5, E22, E5, E5, E11, E11, E9, E9, E11, E9, E...","[E5->E22, E22->E5, E5->E5, E5->E11, E11->E11, ..."
1,blk_7503483334202473044,"[E5,E5,E22,E5,E11,E9,E11,E9,E11,E9,E26,E26,E26...","[E5, E5, E22, E5, E11, E9, E11, E9, E11, E9, E...","[E5->E5, E5->E22, E22->E5, E5->E11, E11->E9, E..."
2,blk_-3544583377289625738,"[E5,E22,E5,E5,E11,E9,E11,E9,E11,E9,E3,E26,E26,...","[E5, E22, E5, E5, E11, E9, E11, E9, E11, E9, E...","[E5->E22, E22->E5, E5->E5, E5->E11, E11->E9, E..."
3,blk_-9073992586687739851,"[E5,E22,E5,E5,E11,E9,E11,E9,E11,E9,E26,E26,E26...","[E5, E22, E5, E5, E11, E9, E11, E9, E11, E9, E...","[E5->E22, E22->E5, E5->E5, E5->E11, E11->E9, E..."
4,blk_7854771516489510256,"[E5,E5,E22,E5,E11,E9,E11,E9,E11,E9,E26,E26,E26...","[E5, E5, E22, E5, E11, E9, E11, E9, E11, E9, E...","[E5->E5, E5->E22, E22->E5, E5->E11, E11->E9, E..."


In [16]:
# Count the occurrences of each two-event sequence

two_event_features["bigram_counts"] = two_event_features["bigrams"].apply(
    Counter
)

two_event_features.head()

,BlockId,Features,event_list,bigrams,bigram_counts
0,blk_-1608999687919862906,"[E5,E22,E5,E5,E11,E11,E9,E9,E11,E9,E26,E26,E26...","[E5, E22, E5, E5, E11, E11, E9, E9, E11, E9, E...","[E5->E22, E22->E5, E5->E5, E5->E11, E11->E11, ...","{'E5->E22': 1, 'E22->E5': 1, 'E5->E5': 2, 'E5-..."
1,blk_7503483334202473044,"[E5,E5,E22,E5,E11,E9,E11,E9,E11,E9,E26,E26,E26...","[E5, E5, E22, E5, E11, E9, E11, E9, E11, E9, E...","[E5->E5, E5->E22, E22->E5, E5->E11, E11->E9, E...","{'E5->E5': 1, 'E5->E22': 1, 'E22->E5': 1, 'E5-..."
2,blk_-3544583377289625738,"[E5,E22,E5,E5,E11,E9,E11,E9,E11,E9,E3,E26,E26,...","[E5, E22, E5, E5, E11, E9, E11, E9, E11, E9, E...","[E5->E22, E22->E5, E5->E5, E5->E11, E11->E9, E...","{'E5->E22': 1, 'E22->E5': 1, 'E5->E5': 1, 'E5-..."
3,blk_-9073992586687739851,"[E5,E22,E5,E5,E11,E9,E11,E9,E11,E9,E26,E26,E26...","[E5, E22, E5, E5, E11, E9, E11, E9, E11, E9, E...","[E5->E22, E22->E5, E5->E5, E5->E11, E11->E9, E...","{'E5->E22': 1, 'E22->E5': 1, 'E5->E5': 1, 'E5-..."
4,blk_7854771516489510256,"[E5,E5,E22,E5,E11,E9,E11,E9,E11,E9,E26,E26,E26...","[E5, E5, E22, E5, E11, E9, E11, E9, E11, E9, E...","[E5->E5, E5->E22, E22->E5, E5->E11, E11->E9, E...","{'E5->E5': 1, 'E5->E22': 1, 'E22->E5': 1, 'E5-..."


#### Feature Engineering: Two-Event Sequence Features

The previous section converted each event trace into an ordered list of individual EventIds. This representation provides the foundation for engineering features that preserve information about the sequence in which events occurred.

Rather than considering only individual event frequencies, this feature set examines every pair of consecutive events that occurred while processing an HDFS block. These ordered two-event sequences, commonly referred to as **bigrams**, capture the local transitions between system events while preserving their chronological order.

For example, the event trace:

`E5 → E22 → E5 → E5`

produces the following ordered two-event sequences:

- `E5 → E22`
- `E22 → E5`
- `E5 → E5`

Each unique two-event sequence is then counted to determine how many times it appears within the trace. These sequence frequencies become additional numerical features that complement the event occurrence counts while preserving information about the local ordering of events.

In [17]:
# Expand each block's bigram counts into individual feature columns

bigram_df = pd.DataFrame(
    two_event_features["bigram_counts"].tolist()
).fillna(0)

bigram_df.head()

,E5->E22,E22->E5,E5->E5,E5->E11,E11->E11,E11->E9,E9->E9,E9->E11,E9->E26,E26->E26,...,E8->E14,E5->E1,E15->E12,E22->E12,E18->E21,E18->E2,E23->E26,E18->E11,E13->E18,E27->E27
0,1.0,1.0,2.0,1.0,1.0,2.0,1.0,1.0,1.0,5.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1.0,1.0,1.0,1.0,0.0,3.0,0.0,2.0,1.0,2.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1.0,1.0,1.0,1.0,0.0,3.0,0.0,2.0,0.0,2.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1.0,1.0,1.0,1.0,0.0,3.0,0.0,2.0,1.0,2.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1.0,1.0,1.0,1.0,0.0,3.0,0.0,2.0,1.0,2.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [18]:
# Combine Feature Set 2 with the engineered bigram features

feature_set_3 = pd.concat(
    [
        feature_set_2.reset_index(drop=True),
        bigram_df.reset_index(drop=True)
    ],
    axis=1
)

feature_set_3.head()

,BlockId,Label,E1,E2,E3,E4,E5,E6,E7,E8,...,E8->E14,E5->E1,E15->E12,E22->E12,E18->E21,E18->E2,E23->E26,E18->E11,E13->E18,E27->E27
0,blk_-1608999687919862906,Success,0,0,203,0,10,7,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,blk_7503483334202473044,Success,0,2,1,0,3,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,blk_-3544583377289625738,Fail,0,0,203,0,3,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,blk_-9073992586687739851,Success,0,3,0,0,3,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,blk_7854771516489510256,Success,0,3,1,15,3,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [19]:
# Display the dimensions of Feature Set 3

print("Rows:", feature_set_3.shape[0])
print("Columns:", feature_set_3.shape[1])

Rows: 575061
Columns: 313


In [20]:
# Define the output location for Feature Set 3

feature_set_3_path = (
    processed_dir / "feature_set_3.csv"
)

# Export the completed feature set

feature_set_3.to_csv(
    feature_set_3_path,
    index=False
)

# Confirm the file was successfully created

print("Feature Set 3 saved to:")
print(feature_set_3_path)

Feature Set 3 saved to:
c:\Users\taman\Documents\hdfs-anomaly-detection-study\data\processed\feature_set_3.csv


#### Feature Set 3 Summary

Feature Set 3 extends the previous feature representation by incorporating ordered two-event sequence frequencies into the engineered dataset.

While Feature Set 2 captures how often individual events occur and summarizes basic trace characteristics, Feature Set 3 additionally records how frequently consecutive pairs of events appear within each HDFS block. These engineered bigram features preserve local ordering information that is not represented by event occurrence counts alone.

The completed feature set contains:

- Event occurrence counts (E1 through E29)
- Trace length
- Number of unique events within each trace)
- Ordered two-event sequence frequency features (bigrams)

Engineering the bigram features produced 282 unique two-event sequence columns across the dataset. Each column represents a distinct ordered transition between two consecutive events observed in at least one HDFS block. Because not every block contains every sequence, many of these feature values are zero, resulting in a sparse feature representation. This sparsity is expected and reflects the fact that each block follows only a subset of the possible event transitions.

The completed Feature Set 3 contains 313 columns and has been exported to the `data/processed/` directory as `feature_set_3.csv`. This feature set will be evaluated alongside the other engineered feature sets to determine whether incorporating local event ordering improves anomaly detection performance compared to using event occurrence counts and basic trace features alone.

### *Feature Set 4: Event Occurrence + Basic Trace Features + Three-Event Sequence Features*

Feature Set 4 builds upon the previous feature sets by incorporating ordered three-event sequences into the feature representation.

While Feature Set 3 captures transitions between pairs of consecutive events, this feature set extends the sequential context by examining groups of three consecutive events, commonly referred to as **trigrams**. These features preserve a larger portion of each HDFS block's original event sequence and may capture processing patterns that are not represented by individual event frequencies or two-event transitions.

For example, the event trace:

`E5 → E22 → E5 → E5 → E11`

produces the following ordered three-event sequences:

- `E5 → E22 → E5`
- `E22 → E5 → E5`
- `E5 → E5 → E11`

Each unique three-event sequence is counted within its corresponding trace. These sequence frequencies become additional numerical features that further enrich the representation of each HDFS block while preserving more of the original event ordering.

In [21]:
# Create a copy of the trace dataset for Feature Set 4

three_event_features = traces_df[
    ["BlockId", "Features"]
].copy()

# Convert each trace into a list of EventIds

three_event_features["event_list"] = (
    three_event_features["Features"]
    .apply(lambda x: x.strip("[]").split(","))
)

three_event_features.head()

,BlockId,Features,event_list
0,blk_-1608999687919862906,"[E5,E22,E5,E5,E11,E11,E9,E9,E11,E9,E26,E26,E26...","[E5, E22, E5, E5, E11, E11, E9, E9, E11, E9, E..."
1,blk_7503483334202473044,"[E5,E5,E22,E5,E11,E9,E11,E9,E11,E9,E26,E26,E26...","[E5, E5, E22, E5, E11, E9, E11, E9, E11, E9, E..."
2,blk_-3544583377289625738,"[E5,E22,E5,E5,E11,E9,E11,E9,E11,E9,E3,E26,E26,...","[E5, E22, E5, E5, E11, E9, E11, E9, E11, E9, E..."
3,blk_-9073992586687739851,"[E5,E22,E5,E5,E11,E9,E11,E9,E11,E9,E26,E26,E26...","[E5, E22, E5, E5, E11, E9, E11, E9, E11, E9, E..."
4,blk_7854771516489510256,"[E5,E5,E22,E5,E11,E9,E11,E9,E11,E9,E26,E26,E26...","[E5, E5, E22, E5, E11, E9, E11, E9, E11, E9, E..."


In [22]:
# Generate every ordered three-event sequence contained within each trace

three_event_features["trigrams"] = (
    three_event_features["event_list"]
    .apply(
        lambda events: [
            f"{events[i]}->{events[i+1]}->{events[i+2]}"
            for i in range(len(events) - 2)
        ]
    )
)

three_event_features.head()

,BlockId,Features,event_list,trigrams
0,blk_-1608999687919862906,"[E5,E22,E5,E5,E11,E11,E9,E9,E11,E9,E26,E26,E26...","[E5, E22, E5, E5, E11, E11, E9, E9, E11, E9, E...","[E5->E22->E5, E22->E5->E5, E5->E5->E11, E5->E1..."
1,blk_7503483334202473044,"[E5,E5,E22,E5,E11,E9,E11,E9,E11,E9,E26,E26,E26...","[E5, E5, E22, E5, E11, E9, E11, E9, E11, E9, E...","[E5->E5->E22, E5->E22->E5, E22->E5->E11, E5->E..."
2,blk_-3544583377289625738,"[E5,E22,E5,E5,E11,E9,E11,E9,E11,E9,E3,E26,E26,...","[E5, E22, E5, E5, E11, E9, E11, E9, E11, E9, E...","[E5->E22->E5, E22->E5->E5, E5->E5->E11, E5->E1..."
3,blk_-9073992586687739851,"[E5,E22,E5,E5,E11,E9,E11,E9,E11,E9,E26,E26,E26...","[E5, E22, E5, E5, E11, E9, E11, E9, E11, E9, E...","[E5->E22->E5, E22->E5->E5, E5->E5->E11, E5->E1..."
4,blk_7854771516489510256,"[E5,E5,E22,E5,E11,E9,E11,E9,E11,E9,E26,E26,E26...","[E5, E5, E22, E5, E11, E9, E11, E9, E11, E9, E...","[E5->E5->E22, E5->E22->E5, E22->E5->E11, E5->E..."


In [23]:
# Count the occurrences of each three-event sequence

three_event_features["trigram_counts"] = (
    three_event_features["trigrams"]
    .apply(Counter)
)

three_event_features.head()

,BlockId,Features,event_list,trigrams,trigram_counts
0,blk_-1608999687919862906,"[E5,E22,E5,E5,E11,E11,E9,E9,E11,E9,E26,E26,E26...","[E5, E22, E5, E5, E11, E11, E9, E9, E11, E9, E...","[E5->E22->E5, E22->E5->E5, E5->E5->E11, E5->E1...","{'E5->E22->E5': 1, 'E22->E5->E5': 1, 'E5->E5->..."
1,blk_7503483334202473044,"[E5,E5,E22,E5,E11,E9,E11,E9,E11,E9,E26,E26,E26...","[E5, E5, E22, E5, E11, E9, E11, E9, E11, E9, E...","[E5->E5->E22, E5->E22->E5, E22->E5->E11, E5->E...","{'E5->E5->E22': 1, 'E5->E22->E5': 1, 'E22->E5-..."
2,blk_-3544583377289625738,"[E5,E22,E5,E5,E11,E9,E11,E9,E11,E9,E3,E26,E26,...","[E5, E22, E5, E5, E11, E9, E11, E9, E11, E9, E...","[E5->E22->E5, E22->E5->E5, E5->E5->E11, E5->E1...","{'E5->E22->E5': 1, 'E22->E5->E5': 1, 'E5->E5->..."
3,blk_-9073992586687739851,"[E5,E22,E5,E5,E11,E9,E11,E9,E11,E9,E26,E26,E26...","[E5, E22, E5, E5, E11, E9, E11, E9, E11, E9, E...","[E5->E22->E5, E22->E5->E5, E5->E5->E11, E5->E1...","{'E5->E22->E5': 1, 'E22->E5->E5': 1, 'E5->E5->..."
4,blk_7854771516489510256,"[E5,E5,E22,E5,E11,E9,E11,E9,E11,E9,E26,E26,E26...","[E5, E5, E22, E5, E11, E9, E11, E9, E11, E9, E...","[E5->E5->E22, E5->E22->E5, E22->E5->E11, E5->E...","{'E5->E5->E22': 1, 'E5->E22->E5': 1, 'E22->E5-..."


In [24]:
# Expand each block's trigram counts into individual feature columns

trigram_df = pd.DataFrame(
    three_event_features["trigram_counts"].tolist()
).fillna(0)

trigram_df.head()

,E5->E22->E5,E22->E5->E5,E5->E5->E11,E5->E11->E11,E11->E11->E9,E11->E9->E9,E9->E9->E11,E9->E11->E9,E11->E9->E26,E9->E26->E26,...,E27->E26->E16,E5->E13->E9,E13->E13->E18,E26->E27->E27,E27->E27->E26,E3->E3->E11,E3->E26->E22,E22->E26->E13,E22->E13->E26,E9->E9->E27
0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1.0,1.0,1.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1.0,1.0,1.0,0.0,0.0,0.0,0.0,2.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [25]:
# Combine Feature Set 2 with the engineered trigram features

feature_set_4 = pd.concat(
    [
        feature_set_2.reset_index(drop=True),
        trigram_df.reset_index(drop=True)
    ],
    axis=1
)

feature_set_4.head()

,BlockId,Label,E1,E2,E3,E4,E5,E6,E7,E8,...,E27->E26->E16,E5->E13->E9,E13->E13->E18,E26->E27->E27,E27->E27->E26,E3->E3->E11,E3->E26->E22,E22->E26->E13,E22->E13->E26,E9->E9->E27
0,blk_-1608999687919862906,Success,0,0,203,0,10,7,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,blk_7503483334202473044,Success,0,2,1,0,3,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,blk_-3544583377289625738,Fail,0,0,203,0,3,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,blk_-9073992586687739851,Success,0,3,0,0,3,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,blk_7854771516489510256,Success,0,3,1,15,3,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [26]:
# Verify Feature Set 4 dimensions

print("Rows:", feature_set_4.shape[0])
print("Columns:", feature_set_4.shape[1])

Rows: 575061
Columns: 1084


In [27]:
# Display the Feature Set 4 columns

feature_set_4.columns.tolist()

['BlockId',
 'Label',
 'E1',
 'E2',
 'E3',
 'E4',
 'E5',
 'E6',
 'E7',
 'E8',
 'E9',
 'E10',
 'E11',
 'E12',
 'E13',
 'E14',
 'E15',
 'E16',
 'E17',
 'E18',
 'E19',
 'E20',
 'E21',
 'E22',
 'E23',
 'E24',
 'E25',
 'E26',
 'E27',
 'E28',
 'E29',
 'trace_length',
 'unique_event_count',
 'E5->E22->E5',
 'E22->E5->E5',
 'E5->E5->E11',
 'E5->E11->E11',
 'E11->E11->E9',
 'E11->E9->E9',
 'E9->E9->E11',
 'E9->E11->E9',
 'E11->E9->E26',
 'E9->E26->E26',
 'E26->E26->E26',
 'E26->E26->E6',
 'E26->E6->E5',
 'E6->E5->E16',
 'E5->E16->E6',
 'E16->E6->E5',
 'E6->E5->E18',
 'E5->E18->E25',
 'E18->E25->E26',
 'E25->E26->E26',
 'E26->E26->E3',
 'E26->E3->E25',
 'E3->E25->E6',
 'E25->E6->E6',
 'E6->E6->E5',
 'E6->E5->E5',
 'E5->E5->E16',
 'E5->E16->E18',
 'E16->E18->E26',
 'E18->E26->E26',
 'E26->E26->E5',
 'E26->E5->E6',
 'E5->E6->E5',
 'E5->E16->E3',
 'E16->E3->E3',
 'E3->E3->E3',
 'E3->E3->E18',
 'E3->E18->E25',
 'E18->E25->E6',
 'E25->E6->E3',
 'E6->E3->E3',
 'E3->E3->E26',
 'E3->E26->E26',
 'E26->E3

In [28]:
# Define the output location for Feature Set 4

feature_set_4_path = (
    processed_dir / "feature_set_4.csv"
)

# Export the feature set to the processed folder

feature_set_4.to_csv(
    feature_set_4_path,
    index=False
)

# Confirm the file was successfully created

print("Feature Set 4 saved to:")
print(feature_set_4_path)

Feature Set 4 saved to:
c:\Users\taman\Documents\hdfs-anomaly-detection-study\data\processed\feature_set_4.csv


#### Feature Set 4 Summary

Feature Set 4 extends the previous feature representations by incorporating ordered three-event sequence frequencies into the engineered dataset.

While Feature Set 3 captures transitions between consecutive pairs of events, Feature Set 4 further preserves the chronological structure of each HDFS block by recording the frequency of every ordered three-event sequence observed within each trace. These engineered trigram features provide additional sequential context that may reveal processing patterns not captured by event occurrence counts or two-event transitions alone.

The completed feature set contains:

- Event occurrence counts (E1 through E29)
- Trace length
- Number of unique events within each trace
- Ordered three-event sequence frequency features (trigrams)

Engineering the trigram features produced 1,053 unique three-event sequence columns across the dataset. Each column represents a distinct ordered sequence of three consecutive events observed in at least one HDFS block. As with the bigram features, many of these values are zero because each block follows only a subset of the possible event sequences. This sparse representation is expected and reflects the diversity of processing paths within the HDFS system.

The completed Feature Set 4 contains 1,084 columns and has been exported to the `data/processed/` directory as `feature_set_4.csv`. This feature set will be evaluated alongside the other engineered feature sets to determine whether incorporating additional sequential context improves anomaly detection performance.

## **Feature Engineering Complete**

The feature engineering phase successfully transformed the original HDFS datasets into multiple machine learning ready feature representations.

Four independent feature sets were engineered, each progressively incorporating additional information about HDFS block processing:

- **Feature Set 1:** Event occurrence counts.
- **Feature Set 2:** Event occurrence counts with basic trace characteristics.
- **Feature Set 3:** Event occurrence counts, basic trace characteristics, and ordered two-event sequence frequencies.
- **Feature Set 4:** Event occurrence counts, basic trace characteristics, and ordered three-event sequence frequencies.

Each feature set was exported as an independent processed dataset and will be evaluated using the same experimental framework. This approach allows the study to determine whether progressively richer feature representations improve anomaly detection performance while holding the learning algorithms constant.

With feature engineering complete, the next phase of the study will focus on preparing the experimental datasets, performing the 80/20 training and testing split, developing the statistical rule-based baseline, and training the supervised and unsupervised machine learning models for comparison.